# ArcFace + ConvNeXt-Tiny Micro Finetune at 384px

This Colab notebook resumes from the champion checkpoint `arcface_convnext_best.pth` stored in Google Drive, loads model and optimizer states, and runs a short 20-epoch finetune at 384 resolution with batch size 16.

Key changes for Phase 3:
- Image size: 384
- Batch size: 16
- Backbone LR: 1e-6
- ArcFace head LR: 1e-5
- Scheduler: CosineAnnealingLR
- Augmentation: light RandomRotation(15)

In [ ]:
# 2) Mount Google Drive and define paths
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/kaplumbaga_tanima'
DATA_ROOT = os.path.join(PROJECT_ROOT, 'datasets')
TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
VAL_DIR = os.path.join(DATA_ROOT, 'test')  # use test/val folder if that is your holdout split
CHECKPOINT_CANDIDATES = [
    os.path.join(PROJECT_ROOT, 'arcface_convnext_best.pth'),
    os.path.join(PROJECT_ROOT, 'checkpoints', 'arcface_convnext_best.pth'),
]
CHECKPOINT_IN = next((path for path in CHECKPOINT_CANDIDATES if os.path.exists(path)), CHECKPOINT_CANDIDATES[0])
CHECKPOINT_OUT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints', 'finetune_384')
LOG_DIR = os.path.join(PROJECT_ROOT, 'logs')
os.makedirs(CHECKPOINT_OUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('TRAIN_DIR exists:', os.path.exists(TRAIN_DIR))
print('VAL_DIR exists:', os.path.exists(VAL_DIR))
print('CHECKPOINT_IN exists:', os.path.exists(CHECKPOINT_IN))

In [ ]:
# 3) Configuration
import random
import numpy as np
import math
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm
import timm

SEED = 42
BATCH_SIZE = 16
IMG_SIZE = 384
RESUME_EPOCHS = 20
BACKBONE_LR = 1e-6
HEAD_LR = 1e-5
WEIGHT_DECAY = 5e-4
EMBEDDING_DIM = 512
ARCFACE_S = 30.0
ARCFACE_M = 0.50
LABEL_SMOOTHING = 0.1
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

print('BATCH_SIZE:', BATCH_SIZE)
print('IMG_SIZE:', IMG_SIZE)
print('RESUME_EPOCHS:', RESUME_EPOCHS)

In [ ]:
# 3) Configuration
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm
import timm

SEED = 42
BATCH_SIZE = 16
IMG_SIZE = 384
RESUME_EPOCHS = 20
BACKBONE_LR = 1e-6
HEAD_LR = 1e-5
WEIGHT_DECAY = 5e-4
EMBEDDING_DIM = 512
ARCFACE_S = 30.0
ARCFACE_M = 0.50
LABEL_SMOOTHING = 0.1
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

print('BATCH_SIZE:', BATCH_SIZE)
print('IMG_SIZE:', IMG_SIZE)
print('RESUME_EPOCHS:', RESUME_EPOCHS)

In [ ]:
# 4) Dataset and dataloaders with light rotation for angled turtles
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = ImageFolder(VAL_DIR, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

num_classes = len(train_dataset.classes)
print('Classes:', num_classes)
print('Train samples:', len(train_dataset))
print('Val samples:', len(val_dataset))

In [ ]:
# 5) Model, ArcFace head, and flexible checkpoint loading
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.5, easy_margin=False, ls_eps=0.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.ls_eps = ls_eps
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.easy_margin = easy_margin
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input_tensor, label):
        cosine = F.linear(F.normalize(input_tensor), F.normalize(self.weight))
        sine = torch.sqrt(torch.clamp(1.0 - torch.pow(cosine, 2), min=0.0, max=1.0))
        phi = cosine * self.cos_m - sine * self.sin_m
        if self.easy_margin:
            phi = torch.where(cosine > 0, phi, cosine)
        else:
            phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1.0)
        if self.ls_eps > 0:
            one_hot = (1 - self.ls_eps) * one_hot + self.ls_eps / self.out_features
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        return output

class TurtleModel(nn.Module):
    def __init__(self, num_classes, model_name='convnext_tiny.fb_in22k_ft_in1k', embedding_dim=512, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        in_features = self.backbone.num_features
        self.bn1 = nn.BatchNorm1d(in_features)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(in_features, embedding_dim)
        self.bn2 = nn.BatchNorm1d(embedding_dim)

    def forward(self, x):
        x = self.backbone(x)
        x = self.bn1(x)
        x = self.dropout(x)
        x = self.fc(x)
        x = self.bn2(x)
        return x

def load_resume_checkpoint(model, metric_fc, optimizer, scheduler, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if 'model_state_dict' not in checkpoint:
        raise KeyError('Checkpoint must contain model_state_dict for resume training.')
    if 'metric_fc_state_dict' not in checkpoint:
        raise KeyError('Checkpoint must contain metric_fc_state_dict for resume training.')
    if 'optimizer_state_dict' not in checkpoint:
        raise KeyError('Checkpoint must contain optimizer_state_dict for exact resume training.')

    model.load_state_dict(checkpoint['model_state_dict'])
    metric_fc.load_state_dict(checkpoint['metric_fc_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if scheduler is not None and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    checkpoint_epoch = int(checkpoint.get('epoch', 20))
    best_acc = float(checkpoint.get('acc', 0.0))
    return checkpoint_epoch, best_acc

In [ ]:
# 6) Build optimizer and scheduler, then resume from Drive checkpoint
model = TurtleModel(num_classes=num_classes, embedding_dim=EMBEDDING_DIM, pretrained=True).to(DEVICE)
metric_fc = ArcMarginProduct(EMBEDDING_DIM, num_classes, s=ARCFACE_S, m=ARCFACE_M, ls_eps=LABEL_SMOOTHING).to(DEVICE)

optimizer = optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': BACKBONE_LR},
    {'params': model.fc.parameters(), 'lr': HEAD_LR},
    {'params': metric_fc.parameters(), 'lr': HEAD_LR},
], weight_decay=WEIGHT_DECAY)

criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=RESUME_EPOCHS)

start_epoch, best_acc = load_resume_checkpoint(
    model=model,
    metric_fc=metric_fc,
    optimizer=optimizer,
    scheduler=scheduler,
    checkpoint_path=CHECKPOINT_IN,
    device=DEVICE,
)

resume_from_epoch = start_epoch + 1
end_epoch = start_epoch + RESUME_EPOCHS
print('Resumed from epoch:', start_epoch)
print('Will train from epoch', resume_from_epoch, 'to', end_epoch)
print('Best acc in checkpoint:', best_acc)

In [ ]:
# 7) Micro-finetuning loop: 20 epochs at 384px
from datetime import datetime

run_name = f"finetune_384_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
run_log_path = os.path.join(LOG_DIR, f'{run_name}.log')
history = []

for epoch in range(resume_from_epoch, end_epoch + 1):
    model.train()
    metric_fc.train()
    running_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(train_loader, total=len(train_loader), leave=False)
    for images, labels in loop:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        embeddings = model(images)
        outputs = metric_fc(embeddings, labels)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        loop.set_description(f'Epoch [{epoch}/{end_epoch}]')
        loop.set_postfix(loss=loss.item(), acc=100.0 * correct / total)

    scheduler.step()
    epoch_loss = running_loss / max(1, len(train_loader))
    epoch_acc = 100.0 * correct / max(1, total)
    current_lr = optimizer.param_groups[0]['lr']

    epoch_msg = f'Epoch {epoch}/{end_epoch} complete. Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.2f}%, LR: {current_lr:.8f}'
    print(epoch_msg)
    history.append(epoch_msg)

    checkpoint = {
        'epoch': epoch,
        'acc': epoch_acc,
        'model_state_dict': model.state_dict(),
        'metric_fc_state_dict': metric_fc.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'num_classes': num_classes,
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
    }

    latest_path = os.path.join(CHECKPOINT_OUT_DIR, 'latest_384.pth')
    torch.save(checkpoint, latest_path)

    if epoch_acc >= best_acc:
        best_acc = epoch_acc
        best_path = os.path.join(CHECKPOINT_OUT_DIR, 'best_384.pth')
        torch.save(checkpoint, best_path)
        print('Saved best checkpoint:', best_path)

    final_path = os.path.join(CHECKPOINT_OUT_DIR, f'epoch_{epoch:03d}.pth')
    torch.save(checkpoint, final_path)

    with open(run_log_path, 'a', encoding='utf-8') as f:
        f.write(epoch_msg + '\n')

print('Finetune complete. Best Acc:', best_acc)
print('Logs written to:', run_log_path)

## Notes
- If the checkpoint does not contain `optimizer_state_dict`, this notebook intentionally stops so resume is exact.
- The learning rate is intentionally tiny for micro-finetuning.
- If Colab resets, remount Drive before running again.
- After training, prefer `best_384.pth` for evaluation and deployment.